 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict, Tuple

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

Updating cache.
/users/60638/items/top?limit=100&start=100
/users/60638/items/top?limit=100&start=200
/users/60638/items/top?limit=100&start=300
/users/60638/items/top?limit=100&start=400
/users/60638/items/top?limit=100&start=500
/users/60638/items/top?limit=100&start=600
/users/60638/items/top?limit=100&start=700
/users/60638/items/top?limit=100&start=800
/users/60638/items/top?limit=100&start=900
/users/60638/items/top?limit=100&start=1000
/users/60638/items/top?limit=100&start=1100
/users/60638/items/top?limit=100&start=1200
/users/60638/items/top?limit=100&start=1300
/users/60638/items/top?limit=100&start=1400
/users/60638/items/top?limit=100&start=1500
/users/60638/items/top?limit=100&start=1600
/users/60638/items/top?limit=100&start=1700


In [2]:
PROMPT_END_STR_PERPLEX = '---'    
RESPONSE_SOURCES_DIVIDER_STR = '<div style="text-align: center">⁂</div>'
source_list_pattern_perplex = re.compile(r'\[\^?(?P<num>\d+)\]:\s*(?P<url>http[s]?://\S+)')

perplex_source_list_OLD_FORMATre = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
# sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)') # note used?

relinker = lpz.relinker

def split_single_prs_text_perplex(pr_text: str) -> Tuple[str, str, str, str]:
    """Splits stock perplexity export markdown text into prompt, response and source sections,
    returning the source information in citenum_url_pairs.  This is the only way
    to associate citenums to urls, as stock perplexity response citenum are markdown footnotes,
    with plain citenums, like this: [citenum] or [^citenum].  These are left plain, as is."""

    match = re.search(r'(?m)^# (?P<heading_text>.+)', pr_text)
    if (heading_start_index := match.start('heading_text')) == -1:
        raise ValueError('Could not find prompt heading')
    
    preamble = pr_text[:heading_start_index].strip()
    
    prompt_end_index, response_start_index = rfw.find_markdown_divider_boundaries(pr_text)
        
    if heading_start_index >= prompt_end_index:
        raise ValueError(f'{heading_start_index=} >= {prompt_end_index=}. '
                         'Probably missed the starting level 1 header part of the prompt.')
    
    prompt = pr_text[heading_start_index:prompt_end_index+1].strip()

    response_sources_divider_index = pr_text.rfind(RESPONSE_SOURCES_DIVIDER_STR)

    if response_sources_divider_index == -1:
        raise ValueError('Could not find divider between AI response and sources list')

    if response_sources_divider_index <= response_start_index:
        raise ValueError('body_sources_divider_index <= response_sources_divider_index')
    
    response = f"{pr_text[response_start_index:response_sources_divider_index]}".strip()
    
    sources = pr_text[response_sources_divider_index:]
    citenum_url_pairs = rfw.get_link_tu_pairs(sources, source_list_pattern_perplex)
    
    return lpz.PromptResponseSplit(preamble, prompt, response, citenum_url_pairs, None) # no source titles

In [3]:
# def split_single_prompt_response_dedup_perplex(markdown_text: str) -> lpz.PromptResponseSplitDeDup:
#     """Splits perplexity output markdown text into prompt, response and source sections.
#     In the response, duplicate citenums are removed, and the mapping from original 
#     to deduplicated numbers is in citenumes_to_url_source"""
#     raise ValueError('should not be calling this anymore')
#     return relinker.split_prompt_response_dedup(markdown_text, split_single_prompt_response_text_perplex)

def relink_single_file_perplexity(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    "Relinks and writes to a file a single prompt/response from perplexity."
    file_text = lpz.read_markdown_file(perplexity_file)
    prsplit = relinker.split_single_prs_dedup(file_text, split_single_prs_text_perplex)
    #prsplit = split_single_prompt_response_dedup_perplex(file_text)
    
    body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(prsplit,'plain_link')
    body_relinked = rfw.hierarch_shift_markdown_headers(body_relinked, top_level=2)
    source_link = rfw.file_link_md('source', perplexity_file)

    relinked_file.write_text(f'{lpz.make_obsidian_front_matter()}\n*{source_link}*\n# Prompt\n\n{prsplit.prompt}\n'
                             f'# Response\n\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', 
                             encoding='utf-8')

In [4]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
#perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
# perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / "perple_new_format_longprompt_example.md"
perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex' / 'GPT-4o.md'

output_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Scratch Space")

output_file = output_dir / "tmp_perplex_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = False
relink_single_file_perplexity(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/GPT-4o.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/tmp_perplex_example.md')


ic| len(dedup_num_to_url): 60
    dedup_num_to_url: {'1': 'https://www.euractiv.com/section/elections/opinion/poland-has-shown-how-to-defeat-populism',
                       '10': 'https://www.populismstudies.org/eight-years-of-populist-rule-in-poland-comes-to-an-end',
                       '11': 'https://revista.drclas.harvard.edu/a-review-of-right-wing-populism-in-latin-america-and-beyond',
                       '12': 'https://library.fes.de/pdf-files/bueros/warschau/21049.pdf',
                       '13': 'https://www.reddit.com/r/labouruk/comments/1djtgx9/centrism_is_a_direct_cause_of_right_wing_populism',
                       '14': 'https://www.epc.eu/content/elections_round-up_v3.pdf',
                       '15': 'https://theloop.ecpr.eu/how-real-is-the-threat-of-the-populist-far-right-in-the-european-parliament-elections',
                       '16': 'https://canopyforum.org/2024/10/03/european-elections-2024-successes-and-failures-of-far-right-political-parties',
      

Done.


### Test merging

In [5]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

# multi-file, same prompt
#chat_files = list(datdir.glob('*.md'))
# multi-file, different prompt
#chat_files = [chat_files[3], pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_example.md")]
# single file
#chat_files = [chat_files[3]]

# single smc file but multiprompt
chat_files = [pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_multi_prompt_savemychatbot_example.md")]


merged_output_file = output_dir / 'tmp_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [6]:
def get_prompt_response(chat_fle):
    file_text = lpz.read_markdown_file(chat_file)

    if not lpz.is_smc_content(file_text):
        # stock perplexity files have only a single prompt-response pair
        return [relinker.split_single_prs_dedup(file_text, split_single_prs_text_perplex)]

    prs_splits = []    
    sections = re.split(rf'(?<=\n){lpz.PROMPT_HEADER_SMC}', file_text)
    for section in sections[1:]:  # Process each user section
        section = f'{lpz.PROMPT_HEADER_SMC}\n{section}' # stick header back on for more certtain matching
        dedup_prs = relinker.split_single_prs_dedup(section, lpz.split_single_prs_text_smc)
        prs_splits.append(dedup_prs)

    return prs_splits

verbose = True

num_chat_files = len(chat_files)
all_prompts, all_responses, all_citenums_to_url = [], [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'Parsing {chat_file.name}')

    for prompt_index, prsplit in enumerate(get_prompt_response(chat_file)):
        print(f'{file_index=}, {prompt_index=}')
        all_prompts.append(prsplit.prompt)
        all_responses.append(prsplit.response_dedup)

        citenum_to_url_df = prsplit.citenum_to_url_df.copy().reset_index()
        citenum_to_url_df[['file_index','chat_file', 'prompt_index']] = file_index, chat_file, prompt_index
        if prsplit.url_to_source_title is not None:
            #print('merging titles')
            citenum_to_url_df = citenum_to_url_df.set_index('url', drop=True)
            citenum_to_url_df['title'] = prsplit.url_to_source_title            
            citenum_to_url_df = citenum_to_url_df.reset_index()
        #display(citenum_to_url_df)
        all_citenums_to_url.append(citenum_to_url_df)
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} total citation numbers')

Parsing perplexity_multi_prompt_savemychatbot_example.md
Sources header(\*\*Sources:\*\*) not in expected place or no source list: Assume no sources.
Malformed Plain citenum [20] appears without URL in response
num_url_pair[0]='1', num_url_pair[1]='https://libanswers.lib.miamioh.edu/stats-faq/faq/343635' in response but not source list
num_url_pair[0]='3', num_url_pair[1]='https://stats.stackexchange.com/questions/81659/mutual-information-versus-correlation' in response but not source list
num_url_pair[0]='17', num_url_pair[1]='https://towardsdatascience.com/how-to-measure-relationship-between-variables-d0606df27fd8' in response but not source list
num_url_pair[0]='7', num_url_pair[1]='https://mattiheino.com/2019/05/10/correlation' in response but not source list
num_url_pair[0]='10', num_url_pair[1]='https://m-clark.github.io/docs/correlationcomparison.pdf' in response but not source list
num_url_pair[0]='9', num_url_pair[1]='https://www.stats.ox.ac.uk/~cucuring/lecture_2_correlations

In [7]:
# Fill in titles when find in other sections (useful for debugging?)
fixed_title_dfs = []
for url, df in all_citenums_to_url.groupby('url'):
    has_no_title = df.title.isna()
    # if 'quantdare' in url:
    #     raise Exception(f'found it {url=}')
    #     break
    #ic(has_no_title.sum(), url)
    if any(has_no_title):
        #print(f'missing titles for {url=}')
        if len(titles := df.title[~has_no_title].unique()) > 1:
            ic(url, titles)
            raise ValueError('Different titles for same URL')
        #ic(titles)
        if len(titles) > 0:
            df = df.fillna({'title': titles[0]})
            #print('df after fix:')
            #display(df)
    fixed_title_dfs.append(df.copy())
all_citenums_to_url = pd.concat(fixed_title_dfs)

# fix at the end to allow possible title fill-in from other responses
all_citenums_to_url['title'] = all_citenums_to_url.title.fillna('NO TITLE: likely bare citenum in response w/ no URL')

#### Make a unified cite number set for the merged document

In [8]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Double sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['dedup_num_int'] = df['dedup_num'].astype(int) # so can sort

url_ranks = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_dedup_num_int=('dedup_num_int', 'mean'),
    mean_prompt_index=('prompt_index', 'mean')
).reset_index()

url_ranks = url_ranks.sort_values(by=['mean_file_index', 'mean_prompt_index', 'mean_dedup_num_int'], 
                                 ascending=True).reset_index(drop=True)

# TODO: can I have unif_num as an int?
url_ranks['unif_num'] = np.arange(1, len(url_ranks) + 1).astype(str) # citenum == rank as string

# Merge back the new citenumes

df = df.merge(url_ranks[['url', 'unif_num']], on='url')

In [9]:
all_citenums_to_url = (df.sort_values(by='unif_num', key=lambda col: col.astype(int))
                       .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                       .drop('dedup_num_int', axis=1)
                       .set_index('file_index'))

In [10]:
all_citenums_to_url

,url,orig_num,dedup_num,chat_file,prompt_index,title,unif_num
file_index,,,,,,,
0,https://BARE_CITE_NUMBER_20_IN_RESPONSE_WITH_N...,20,9,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,0,NO TITLE: likely bare citenum in response w/ n...,1
0,https://quantdare.com/what-is-mutual-information,12,4,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1,What is Mutual Information? - Quantdare,2
0,https://quantdare.com/what-is-mutual-information,12,4,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,2,What is Mutual Information? - Quantdare,2
0,https://quantdare.com/what-is-mutual-information,4,4,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1,What is Mutual Information? - Quantdare,2
0,https://quantdare.com/what-is-mutual-information,12,3,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,0,What is Mutual Information? - Quantdare,2
...,...,...,...,...,...,...,...
0,https://stats.stackexchange.com/questions/8165...,3,22,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1,Cite in response but no entry in sources list,27
0,https://en.wikipedia.org/wiki/total_correlation,1,20,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,2,Cite in response but no entry in sources list,28
0,http://www.mathemafrica.org?p=16127,5,22,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,2,Cite in response but no entry in sources list,29


#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [11]:
all_prompts_same = True
for i in range(0,len(all_prompts)-1):
    is_same = all_prompts[i].strip().lower() == all_prompts[i+1].strip().lower()
    all_prompts_same &= is_same

In [12]:
# Replace response dedup_num with unif_num, merge prompt/response pairs into single string
is_multi_file_chat = num_chat_files > 1
do_single_top_prompt = all_prompts_same and is_multi_file_chat

full_prompt_indices = rfw.unique_rows(all_citenums_to_url.reset_index(),['file_index','prompt_index'])

all_citenums_to_url = all_citenums_to_url.reset_index().set_index(['file_index', 'prompt_index'])

all_citenums_to_url

url  \
file_index prompt_index                                                      
0          0             https://BARE_CITE_NUMBER_20_IN_RESPONSE_WITH_N...   
           1              https://quantdare.com/what-is-mutual-information   
           2              https://quantdare.com/what-is-mutual-information   
           1              https://quantdare.com/what-is-mutual-information   
           0              https://quantdare.com/what-is-mutual-information   
...                                                                    ...   
           1             https://stats.stackexchange.com/questions/8165...   
           2               https://en.wikipedia.org/wiki/total_correlation   
           2                           http://www.mathemafrica.org?p=16127   
           2                              https://arxiv.org/abs/2011.04794   
           2             https://proceedings.mlr.press/v206/bai23a/bai2...   

                        orig_num dedup_num  \
file_index prompt_index                      
0          0                  20         9   
           1                  12         4   
           2                  12         4   
           1                   4         4   
           0                  12         3   
...                          ...       ...   
           1                   3        22   
           2                   1        20   
           2                   5        22   
           2                   3        23   
           2                   6        24   

                                                                 chat_file  \
file_index prompt_index                                                      
0          0             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           1             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           2             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           1             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           0             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
...                                                                    ...   
           1             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           2             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           2             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           2             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   
           2             C:\Users\scott\OneDrive\share\ref\refwrangle\t...   

                                                                     title  \
file_index prompt_index                                                      
0          0             NO TITLE: likely bare citenum in response w/ n...   
           1                       What is Mutual Information? - Quantdare   
           2                       What is Mutual Information? - Quantdare   
           1                       What is Mutual Information? - Quantdare   
           0                       What is Mutual Information? - Quantdare   
...                                                                    ...   
           1                 Cite in response but no entry in sources list   
           2                 Cite in response but no entry in sources list   
           2                 Cite in response but no entry in sources list   
           2                 Cite in response but no entry in sources list   
           2                 Cite in response but no entry in sources list   

                        unif_num  
file_index prompt_index           
0          0                   1  
           1                   2  
           2                   2  
           1                   2  
           0                   2  
...                          ...  
           1                  27  
           2                  28  
           2                  29  
           2                  30  
       

In [13]:
all_citenums_to_url.unif_num.unique()

array(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23',
       '24', '25', '26', '27', '28', '29', '30', '31'], dtype=object)

In [14]:

all_promptresp, chat_source_file_link = '', []
for response_index, (file_index, prompt_index) in full_prompt_indices.iterrows():
    if verbose:
        print(f'Unifying {chat_files[file_index].stem}, {prompt_index=}')
        
    # remap deduped citenums to unified citenums
    citenums_to_url_this = all_citenums_to_url.loc[file_index, prompt_index].copy()
    citenums_dedup_to_unified = citenums_to_url_this.set_index('dedup_num').unif_num.to_dict()
    response_unified = relinker.replace_response_citenums(all_responses[response_index], citenums_dedup_to_unified) # unified citenums

    # Put this body within the appropriate merged dialog headings
    source_link = rfw.file_link_md('source', chat_files[file_index])
    if is_multi_file_chat:
        if do_single_top_prompt:
            if file_index == 0:
                all_promptresp += f'# Prompt\n\n{all_prompts[file_index]}\n# Responses\n'
            all_promptresp += f'\n## {chat_files[file_index].name}\n*{source_link}*\n\n'
            all_promptresp += rfw.hierarch_shift_markdown_headers(response_unified, top_level=3)            
        else:
            short_prompt = rfw.get_first_n_words(all_prompts[file_index], lpz.MAX_WORDS_PROMPT_HEADING)
            all_promptresp += f'\n# {short_prompt}\n*{source_link}*\n\n{all_prompts[file_index]}\n## Response\n\n'
            all_promptresp += rfw.hierarch_shift_markdown_headers(response_unified, top_level=3)
    else:
        all_promptresp += f'*{source_link}*\n\n# Prompt\n\n{all_prompts[file_index]}\n# Response\n'
        all_promptresp += response_unified

Unifying perplexity_multi_prompt_savemychatbot_example, prompt_index=0
Unifying perplexity_multi_prompt_savemychatbot_example, prompt_index=1
Unifying perplexity_multi_prompt_savemychatbot_example, prompt_index=2


C:\Users\scott\AppData\Local\Temp\ipykernel_11140\97191270.py:7: PerformanceWarning: indexing past lexsort depth may impact performance.
  citenums_to_url_this = all_citenums_to_url.loc[file_index, prompt_index].copy()


##### Insert links to Obsidian or Zotero

In [ ]:
# unif_num_to_url = all_citenums_to_url[['unif_num', 'url']].drop_duplicates()
# unif_num_to_url.index = unif_num_to_url['unif_num']

# TODO: do better: automate ? handle mixed file types? make all num_dedup plain_links and avoid the problem?
#response_link_type = 'plain_link' # this assumes stock perplexity prompt/resp
response_link_type = 'url_link' # this assumes ALL SMC

# Make the expected function parameter structure
#df = rfw.unique_rows(all_citenums_to_url.reset_index(),['url','title'])

url_to_source_title = {}
for unif_num, df in all_citenums_to_url.reset_index().groupby('unif_num'):
    url_to_source_title[df.iloc[0].url] = df.iloc[0].title

prsplit_all = lpz.PromptResponseSplitDeDup('','', all_promptresp, all_citenums_to_url, url_to_source_title)


# TODO:
# 31 source to in through url_to_source_title but relinked_sources has only 26
# relink_body_and_make_source_links(self, prsplit: PromptResponseSplitDeDup, citenum_col: str = 'dedup_num') -> Tuple[str, list[str]]:

all_promptresp_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(prsplit_all, response_link_type,'unif_num')
#all_promptresp_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(all_promptresp_unified , unified_citenums, 'plain_link')
relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(lpz.citenum_plain_re, line).group('num'))))

print(f'writing to {merged_output_file=}')
merged_output_file.write_text(f'{lpz.make_obsidian_front_matter()}\n{all_promptresp_unified_relinked}\n# Citations\n{relinked_sources}',
                              encoding='utf-8')
print("Done.")

ic| len(dedup_num_to_url): 24
    dedup_num_to_url: {'1': 'https://en.wikipedia.org/wiki/mutual_information',
                       '11': 'https://www.quantiki.org/wiki/mutual-information',
                       '12': 'https://pages.stern.nyu.edu/~dbackus/bcz/entropy/mutual-information-wikipedia.pdf',
                       '13': 'https://www.stat.berkeley.edu/~brill/papers/bjps1.pdf',
                       '14': 'https://mkowal2.github.io/posts/2020/01/understanding-mi',
                       '15': 'https://lcalem.github.io/blog/2018/10/17/mutual-information',
                       '16': 'https://math.stackexchange.com/questions/3020611/how-to-calculate-mutual-information',
                       '17': 'http://www.ece.tufts.edu/ee/194nit/lect01.pdf',
                       '18': 'https://www.youtube.com/embed/ejip_mgvlwe?autoplay=1&color=white&playsinline=true&enablejsapi=1&origin=https%3a%2f%2fwww.perplexity.ai&widgetid=1',
                       '19': 'https://cstheory.stackexc

writing to merged_output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/tmp_perplexy_merged.md')
Done.


In [ ]:

# # df = all_citenums_to_url
# # colNm = 'unif_num'
# # is_numeric = pd.to_numeric(df[colNm], errors='coerce').notnull().all()

# column_names = ['unif_num','url']
# # do_sort=True

# # def unique_rows(df: pd.DataFrame, column_names: list = None) -> pd.DataFrame:
# #     """Returns a sorted dataframe of unique rows in the rows, combo_col_names.
# #     Returned columns will be in order of column_names if given, numerically,
# #     when possible, and the index will be reset."""

# #     cols = list(df.columns) if column_names is None else column_names

# #     # new index = 0:(len(df)-1) (avoids non-unique index problems)
# #     df_orig_cols = df[cols].copy().reset_index(drop=True) 

# #     # sort numerically when possible
# #     df_sort = df_orig_cols.copy()
# #     for col_non_num in df_sort.select_dtypes(exclude=np.number).columns:
# #         col_dat_to_numeric = pd.to_numeric(df_sort[col_non_num], errors='coerce')
# #         if col_dat_to_numeric.notnull().all():
# #             df_sort[col_non_num] = col_dat_to_numeric

# #     index_unique = df_sort.sort_values(cols).drop_duplicates().index
        
# #     return df_orig_cols.loc[index_unique].reset_index(drop=True) 

# dfout = rfw.unique_rows(all_citenums_to_url,column_names)
# dfout.dtypes


unif_num    object
url         object
dtype: object